# Course 1: From Beginner to an AI Research Lab
## Session 2: The math you actually use

**Goal:** fit a line with gradient descent in NumPy, check the gradient numerically, and connect the loss to maximum likelihood.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. Data, the loss and its gradient
Type this yourself rather than running it blindly. The model is `y = w x + b`. The loss is the mean squared error.

```
          n
L(w,b) = (1/n) Σ (w·x_i + b − y_i)²
         i=1
```

Its gradient, by the chain rule, is `dL/dw = 2 · mean(r · x)` and `dL/db = 2 · mean(r)` with `r = w x + b − y`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.uniform(-2, 2, 200)
y = 3 * x + 2 + rng.normal(0, 0.5, 200)

def loss(w, b):
    return np.mean((w * x + b - y) ** 2)

def grad(w, b):
    r = w * x + b - y
    return 2 * np.mean(r * x), 2 * np.mean(r)

### 2. Gradient descent

In [ ]:
import matplotlib.pyplot as plt
w, b, lr = 0.0, 0.0, 0.1
history = []
for step in range(200):
    gw, gb = grad(w, b)
    w, b = w - lr * gw, b - lr * gb
    history.append(loss(w, b))
print(f"w = {w:.3f}, b = {b:.3f}, final loss = {loss(w, b):.3f}")
plt.plot(history); plt.xlabel("step"); plt.ylabel("loss"); plt.yscale("log"); plt.savefig("loss.png")
assert abs(w - 2.97) < 0.02 and abs(b - 1.96) < 0.02 and abs(loss(w, b) - 0.26) < 0.02

The final loss is close to 0.25, the variance of the noise we added. No model can do better than the noise.

### 3. Check the gradient numerically
Compare the formula with a finite difference, at a point away from the optimum.

In [ ]:
h = 1e-5
numeric = (loss(1 + h, 0.5) - loss(1 - h, 0.5)) / (2 * h)
analytic = grad(1, 0.5)[0]
rel = abs(numeric - analytic) / max(abs(numeric), abs(analytic))
print("numeric", numeric, "| analytic", analytic, "| relative error", rel)
assert rel < 1e-8

### 4. Compare with the closed-form answer

In [ ]:
w_ls, b_ls = np.polyfit(x, y, 1)
print(f"np.polyfit: w = {w_ls:.3f}, b = {b_ls:.3f}")
assert abs(w - w_ls) < 1e-3 and abs(b - b_ls) < 1e-3

### 5. Break it on purpose
A learning rate that is too large, and a gradient with a missing factor of 2.

In [ ]:
w2, b2 = 0.0, 0.0
for step in range(20):
    gw, gb = grad(w2, b2); w2, b2 = w2 - 1.0 * gw, b2 - 1.0 * gb               # learning rate 1.0
print("learning rate 1.0, loss after 20 steps:", f"{loss(w2, b2):.3g}", "(it diverged)")

def wrong_grad(w, b):
    r = w * x + b - y; return np.mean(r * x), np.mean(r)                         # the factor of 2 is missing
rel_wrong = abs(numeric - wrong_grad(1, 0.5)[0]) / max(abs(numeric), abs(wrong_grad(1, 0.5)[0]))
print("relative error with the wrong gradient:", round(rel_wrong, 3), "(the check catches it)")
assert loss(w2, b2) > 100 and rel_wrong > 0.4

Notice that gradient descent with the wrong gradient would still have found the right line here, only more slowly. Bugs that slow training without breaking it are common, which is why the numerical check is worth the minute it takes.

### 6. Why squared error? Maximum likelihood
If the noise is Gaussian, the line that makes the data most probable is the line that minimizes the squared error. Check it: the negative log-likelihood and the mean squared error have their minimum at the same w.

In [ ]:
ws = np.linspace(2.5, 3.5, 201); sigma = 0.5
mse = np.array([loss(wi, b) for wi in ws])
nll = np.array([np.sum(0.5 * ((wi * x + b - y) / sigma) ** 2 + np.log(sigma * np.sqrt(2 * np.pi))) for wi in ws])
print("w that minimizes the squared error:", ws[mse.argmin()].round(3), "| w that maximizes the likelihood:", ws[nll.argmin()].round(3))
assert mse.argmin() == nll.argmin()

### TODO
1. Add a quadratic term `a x²` to the model. Derive `dL/da` by hand, then confirm it with the numerical check.
2. Plot the loss as a contour over (w, b) and draw the path gradient descent takes.
3. Replace the Gaussian noise with a few large outliers. Which loss would be more robust, and what noise model does it correspond to?